# Verificación del simulador SP

Antes de sacar conclusiones con el simulador conviene comprobar que **calcula bien** y que
sus **barras de error dicen la verdad**. Este notebook corre esas pruebas; todas deben
decir **OK**. Tarda alrededor de un minuto.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sp_motor as sp

x = sp.estaciones(5)                       # 61 estaciones cada 5 m
xf = np.linspace(0, 300, 6001)             # malla fina
ok = lambda c: "OK   " if c else "FALLA"
print("Estaciones:", len(x))

## 1. Linealidad en k y superposición

V es proporcional a k, y la anomalía de varios cuerpos es la suma de las anomalías de cada
uno. Esto último supone que los cuerpos **no interactúan**: es la aproximación habitual en
SP, válida cuando están separados frente a su tamaño.

In [ ]:
b1 = dict(q=1.5, x0=120, h=10, th=40, k=-15000)
b2 = dict(q=0.5, x0=200, h=8, th=70, k=-80)
print(ok(np.allclose(sp.sp_cuerpo({**b1, "k": 2 * b1["k"]}, x), 2 * sp.sp_cuerpo(b1, x))), "lineal en k")
print(ok(np.allclose(sp.sp_total([b1, b2], x), sp.sp_cuerpo(b1, x) + sp.sp_cuerpo(b2, x))), "superposición")

## 2. Simetría según θ

Con polarización vertical (θ = 90°) la anomalía es simétrica respecto de x₀; con
polarización horizontal (θ = 0°) es antisimétrica.

In [ ]:
for q in (0.5, 1.0, 1.5):
    b = dict(q=q, x0=150, h=10, th=90, k=-100)
    d = np.array([3.0, 17.0, 40.0])
    simetrica = np.allclose(sp.sp_cuerpo(b, 150 + d), sp.sp_cuerpo(b, 150 - d))
    antisim = np.allclose(sp.sp_cuerpo(dict(b, th=0), 150 + d), -sp.sp_cuerpo(dict(b, th=0), 150 - d))
    print(ok(simetrica and antisim), "q = %.1f" % q)

## 3. Amplitud y ancho: lo que permite pasar de la curva al cuerpo

Sobre el cuerpo, $V(x_0) = k\,\sin\theta\,h^{1-2q}$; y con θ = 90° el ancho a media altura
es $2h\sqrt{2^{1/q}-1}$, es decir **1.53 h** (esfera), **2.00 h** (cilindro horizontal) y
**3.46 h** (cilindro vertical).

In [ ]:
for q in (1.5, 1.0, 0.5):
    b = dict(q=q, x0=150, h=12, th=90, k=-100)
    V = np.abs(sp.sp_cuerpo(b, xf))
    sobre = xf[V >= V.max() / 2]
    ancho = sobre.max() - sobre.min()
    print(ok(np.isclose(sp.sp_cuerpo(b, np.array([150.0]))[0], sp.amplitud_en_x0(b))
             and abs(ancho - sp.ancho_media_altura(q, 12)) < 0.1),
          "q = %.1f: ancho a media altura %.2f m = %.2f h" % (q, ancho, ancho / 12))

## 4. La inversión recupera un modelo conocido

Se generan datos **sin ruido** con 1, 2 y 3 cuerpos al azar y se invierte desde un arranque
perturbado. Si el método funciona, debe recuperar la profundidad casi exacta.

In [ ]:
rng = np.random.default_rng(11)
for n in (1, 2, 3):
    exitos = 0
    for prueba in range(10):
        verdad = sp.nuevo_reto(n, semilla=100 * n + prueba)
        obs = sp.observar(verdad, x, 0.0, 0)
        inicio = [dict(b, x0=b["x0"] + rng.normal(0, 15), h=b["h"] * np.exp(rng.normal(0, 0.3)),
                       th=np.clip(b["th"] + rng.normal(0, 20), -89, 89)) for b in verdad]
        sp.optimizar_k(inicio, x, obs)
        r = sp.invertir(inicio, x, obs, sigma=0.0, n_inicios=8)
        error = max(abs(a["h"] - b["h"]) / b["h"] for a, b in zip(r["cuerpos"], verdad))
        exitos += error < 0.01
    print(ok(exitos >= 9), "%d cuerpo(s): h recuperada al 1 %% en %d de 10 casos" % (n, exitos))

## 5. ¿Las barras de error dicen la verdad? (Monte Carlo)

Se repite la inversión con **200 realizaciones distintas del ruido** (σ = 3 mV) sobre el mismo
cuerpo. Si la incertidumbre que reporta el simulador es honesta:

- la dispersión real de las profundidades estimadas debe parecerse al error reportado, y
- cerca del **68 %** de las veces el valor real debe quedar dentro de ±1σ.

In [ ]:
casos = {"esfera": [dict(q=1.5, x0=140.0, h=12.0, th=50.0, k=-20000.0)],
         "2 cuerpos": [dict(q=1.0, x0=100.0, h=10.0, th=70.0, k=-1500.0),
                       dict(q=0.5, x0=210.0, h=8.0, th=30.0, k=-90.0)]}
mc = {}
for nombre, verdad in casos.items():
    est, err = [], []
    for i in range(200):
        obs = sp.observar(verdad, x, 3.0, semilla=1000 + i)
        r = sp.invertir([dict(b) for b in verdad], x, obs, 3.0, n_inicios=1)
        est.append([c["h"] for c in r["cuerpos"]])
        err.append([r["err"][4 * j + 1] for j in range(len(verdad))])
    est, err = np.array(est), np.array(err)
    mc[nombre] = (verdad, est)
    for j, b in enumerate(verdad):
        dentro = np.mean(np.abs(est[:, j] - b["h"]) <= err[:, j])
        print(ok(0.6 <= dentro <= 0.76),
              "%s, cuerpo %d: dispersión real %.2f m | error reportado %.2f m | dentro de ±1σ: %.0f %%"
              % (nombre, j + 1, est[:, j].std(), err[:, j].mean(), 100 * dentro))

verdad, est = mc["esfera"]
plt.figure(figsize=(6, 3.2))
plt.hist(est[:, 0], bins=25, color="#0a6c73", alpha=.8)
plt.axvline(verdad[0]["h"], color="k", ls="--", label="h real")
plt.xlabel("h estimada (m)"); plt.ylabel("realizaciones"); plt.legend()
plt.title("Esfera: 200 inversiones con ruido distinto"); plt.show()

## 6. ¿Se puede identificar la forma?

Para datos de cada forma (σ = 3 mV) se invierte con las tres formas posibles y se comparan
los χ². La forma verdadera debería ganar, pero mira **por cuánto**: si la diferencia es menor
que el margen del azar (≈ 0.4), en un caso real no se podría decidir.

In [ ]:
aciertos, total = 0, 0
for q in (0.5, 1.0, 1.5):
    for sem in range(3):
        verdad = sp.nuevo_reto(1, semilla=500 + sem, formas=[q])
        obs = sp.observar(verdad, x, 3.0, semilla=sem)
        inicio = [dict(q=1.5, x0=verdad[0]["x0"] + 10, h=12, th=45, k=-15000)]
        res = sp.probar_formas(inicio, x, obs, 3.0, n_inicios=6)
        chis = {c[0][0]: c[1] for c in res}
        total += 1
        aciertos += res[0][0][0] == q
        print("verdad q = %.1f → mejor q = %.1f | χ² reducido: %s"
              % (q, res[0][0][0], ", ".join("q%.1f = %.2f" % kv for kv in sorted(chis.items()))))
print(ok(aciertos == total), "forma correcta en %d de %d casos (margen del azar: %.2f)"
      % (aciertos, total, sp.margen_chi2(len(x) - 4)))

## Qué se puede concluir

- El modelo directo cumple las propiedades analíticas de la fórmula: linealidad,
  superposición, simetrías, amplitud y anchos.
- La inversión recupera modelos conocidos y sus barras de error son **honestas**: ~68 % de
  cobertura a 1σ en el Monte Carlo.
- La forma **se puede** identificar con ruido moderado, pero a veces por un margen pequeño.
  Con más ruido o menos estaciones esa diferencia desaparece, y entonces **la forma no está
  determinada por los datos**: esa es la ambigüedad q–h que se discute en clase.

Lo que el simulador **no** verifica: que la fórmula de cuerpos regulares describa bien un
cuerpo real (forma irregular, medio no homogéneo, topografía). Eso es una limitación del
modelo, no del código.

**Ojo con el cilindro vertical (q = 0.5).** Con θ lejos de 90° su anomalía **no vuelve a
cero** en los extremos del perfil: tiende a ±k cos θ, un escalón. No es un error: la fórmula
representa un cuerpo que se prolonga indefinidamente hacia abajo. En datos reales ese escalón
se puede confundir con una deriva o un nivel regional, así que conviene no atribuir a un
cilindro vertical un desnivel que podría ser de fondo.